# Sentiment Analysis of Amazon Reviews

In [35]:
# Import libraries
import pandas as pd
import spacy
from spacytextblob.spacytextblob import SpacyTextBlob

In [36]:
# Load data
df = pd.read_csv(
    "Datafiniti_Amazon_Consumer_Reviews_of_Amazon_Products_May19.csv"
)
df.head()

,id,dateAdded,dateUpdated,name,asins,brand,categories,primaryCategories,imageURLs,keys,...,reviews.didPurchase,reviews.doRecommend,reviews.id,reviews.numHelpful,reviews.rating,reviews.sourceURLs,reviews.text,reviews.title,reviews.username,sourceURLs
0,AVpgNzjwLJeJML43Kpxn,2015-10-30T08:59:32Z,2019-04-25T09:08:16Z,AmazonBasics AAA Performance Alkaline Batterie...,"B00QWO9P0O,B00LH3DMUO",Amazonbasics,"AA,AAA,Health,Electronics,Health & Household,C...",Health & Beauty,https://images-na.ssl-images-amazon.com/images...,"amazonbasics/hl002619,amazonbasicsaaaperforman...",...,NaN,NaN,NaN,NaN,3,https://www.amazon.com/product-reviews/B00QWO9...,I order 3 of them and one of the item is bad q...,... 3 of them and one of the item is bad quali...,Byger yang,"https://www.barcodable.com/upc/841710106442,ht..."
1,AVpgNzjwLJeJML43Kpxn,2015-10-30T08:59:32Z,2019-04-25T09:08:16Z,AmazonBasics AAA Performance Alkaline Batterie...,"B00QWO9P0O,B00LH3DMUO",Amazonbasics,"AA,AAA,Health,Electronics,Health & Household,C...",Health & Beauty,https://images-na.ssl-images-amazon.com/images...,"amazonbasics/hl002619,amazonbasicsaaaperforman...",...,NaN,NaN,NaN,NaN,4,https://www.amazon.com/product-reviews/B00QWO9...,Bulk is always the less expensive way to go fo...,... always the less expensive way to go for pr...,ByMG,"https://www.barcodable.com/upc/841710106442,ht..."
2,AVpgNzjwLJeJML43Kpxn,2015-10-30T08:59:32Z,2019-04-25T09:08:16Z,AmazonBasics AAA Performance Alkaline Batterie...,"B00QWO9P0O,B00LH3DMUO",Amazonbasics,"AA,AAA,Health,Electronics,Health & Household,C...",Health & Beauty,https://images-na.ssl-images-amazon.com/images...,"amazonbasics/hl002619,amazonbasicsaaaperforman...",...,NaN,NaN,NaN,NaN,5,https://www.amazon.com/product-reviews/B00QWO9...,Well they are not Duracell but for the price i...,... are not Duracell but for the price i am ha...,BySharon Lambert,"https://www.barcodable.com/upc/841710106442,ht..."
3,AVpgNzjwLJeJML43Kpxn,2015-10-30T08:59:32Z,2019-04-25T09:08:16Z,AmazonBasics AAA Performance Alkaline Batterie...,"B00QWO9P0O,B00LH3DMUO",Amazonbasics,"AA,AAA,Health,Electronics,Health & Household,C...",Health & Beauty,https://images-na.ssl-images-amazon.com/images...,"amazonbasics/hl002619,amazonbasicsaaaperforman...",...,NaN,NaN,NaN,NaN,5,https://www.amazon.com/product-reviews/B00QWO9...,Seem to work as well as name brand batteries a...,... as well as name brand batteries at a much ...,Bymark sexson,"https://www.barcodable.com/upc/841710106442,ht..."
4,AVpgNzjwLJeJML43Kpxn,2015-10-30T08:59:32Z,2019-04-25T09:08:16Z,AmazonBasics AAA Performance Alkaline Batterie...,"B00QWO9P0O,B00LH3DMUO",Amazonbasics,"AA,AAA,Health,Electronics,Health & Household,C...",Health & Beauty,https://images-na.ssl-images-amazon.com/images...,"amazonbasics/hl002619,amazonbasicsaaaperforman...",...,NaN,NaN,NaN,NaN,5,https://www.amazon.com/product-reviews/B00QWO9...,These batteries are very long lasting the pric...,... batteries are very long lasting the price ...,Bylinda,"https://www.barcodable.com/upc/841710106442,ht..."


In [37]:
# Show dataset information
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 28332 entries, 0 to 28331
Data columns (total 24 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   id                   28332 non-null  str    
 1   dateAdded            28332 non-null  str    
 2   dateUpdated          28332 non-null  str    
 3   name                 28332 non-null  str    
 4   asins                28332 non-null  str    
 5   brand                28332 non-null  str    
 6   categories           28332 non-null  str    
 7   primaryCategories    28332 non-null  str    
 8   imageURLs            28332 non-null  str    
 9   keys                 28332 non-null  str    
 10  manufacturer         28332 non-null  str    
 11  manufacturerNumber   28332 non-null  str    
 12  reviews.date         28332 non-null  str    
 13  reviews.dateSeen     28332 non-null  str    
 14  reviews.didPurchase  9 non-null      object 
 15  reviews.doRecommend  16086 non-null  object 
 1

In [38]:
# Show column names
df.columns

Index(['id', 'dateAdded', 'dateUpdated', 'name', 'asins', 'brand',
       'categories', 'primaryCategories', 'imageURLs', 'keys', 'manufacturer',
       'manufacturerNumber', 'reviews.date', 'reviews.dateSeen',
       'reviews.didPurchase', 'reviews.doRecommend', 'reviews.id',
       'reviews.numHelpful', 'reviews.rating', 'reviews.sourceURLs',
       'reviews.text', 'reviews.title', 'reviews.username', 'sourceURLs'],
      dtype='str')

In [39]:
# Load spaCy medium English model
nlp = spacy.load("en_core_web_md")

# Add TextBlob sentiment component
nlp.add_pipe("spacytextblob")

Preprocessing


In [40]:
# Select review text column
reviews_data = df["reviews.text"]

# Remove rows with missing review text
clean_data = df.dropna(subset=["reviews.text"])

# Check amount of rows in column now
print("Number of reviews:", len(reviews_data))

Number of reviews: 28332


In [41]:
# Create function to clean text of each review
def preprocess_text(text):

    # Convert review to string
    text = str(text)

    # Remove leading and trailing spaces
    text = text.strip()

    # Convert text to lowercase
    text = text.lower()

    # Process text with spaCy
    tokens = nlp(text)

    cleaned_tokens = []

    # Remove stop words and punctuation
    for token in tokens:

        if not token.is_stop and not token.is_punct:
            cleaned_tokens.append(token.text)

    cleaned_text = " ".join(cleaned_tokens)

    return cleaned_text

In [42]:
# Create function to analyse sentiment
def analyse_sentiment(review):

    cleaned_review = preprocess_text(review)

    token = nlp(cleaned_review)

    polarity = token._.blob.polarity

    sentiment = token._.blob.sentiment

    if polarity > 0:
        sentiment_label = "Positive"

    elif polarity < 0:
        sentiment_label = "Negative"

    else:
        sentiment_label = "Neutral"

    return polarity, sentiment_label, sentiment

In [43]:
# Test the model on sample reviews
sample_reviews = reviews_data.head()

for review in sample_reviews:

    polarity, sentiment_label, sentiment = analyse_sentiment(review)

    # Print review sentiments generated by the model
    print("*" * 75)
    print("Review:")
    print(review)

    print("\nPolarity Score:")
    print(polarity)

    print("\nPredicted Sentiment:")
    print(sentiment_label)

    print("\nSentiment Details:")
    print(sentiment)

***************************************************************************
Review:
I order 3 of them and one of the item is bad quality. Is missing backup spring so I have to put a pcs of aluminum to make the battery work.

Polarity Score:
-0.44999999999999996

Predicted Sentiment:
Negative

Sentiment Details:
Sentiment(polarity=-0.44999999999999996, subjectivity=0.35833333333333334)
***************************************************************************
Review:
Bulk is always the less expensive way to go for products like these

Polarity Score:
-0.5

Predicted Sentiment:
Negative

Sentiment Details:
Sentiment(polarity=-0.5, subjectivity=0.7)
***************************************************************************
Review:
Well they are not Duracell but for the price i am happy.

Polarity Score:
0.8

Predicted Sentiment:
Positive

Sentiment Details:
Sentiment(polarity=0.8, subjectivity=1.0)
***************************************************************************
Review:
Seem 

In [44]:
# Compare 2 reviews to see whether they are
# similar in sentiment or not

review_1 = str(reviews_data.iloc[0])

review_2 = str(reviews_data.iloc[1])

token1 = nlp(review_1)
token2 = nlp(review_2)

similarity_score = token1.similarity(token2)

# Print similarity score for reviews
print(f"Similarity Score:", similarity_score)

Similarity Score: 0.951943576335907


As shown above, we can also see whether reviews are similar in sentiment or not. When we print the first 5 reviews and look at their sentiment, they are both negative. The comparison above shows that they are very similar (similarity score close to 1), so it makes sense that they would have similar sentiment.

We can also see that review 3 is positive and has a positive polarity score, so when comparing it to a negative review similarity should be low.

In [45]:
# Compare a positive and negative review
review_1 = str(reviews_data.iloc[0])

review_3 = str(reviews_data.iloc[2])

token1 = nlp(review_1)
token3 = nlp(review_3)

similarity_score = token1.similarity(token3)

# Print similarity score for reviews
print(f"Similarity Score:", similarity_score)

Similarity Score: 0.9200136065483093


As we can see, similarity score is still high although sentiment is not shared. This is indicative of the fact that there can still be improvements.